# Global warming: trends, hiatuses, and polar amplification

In this notebook we use the **Berkeley Earth gridded global temperature dataset** to investigate how global temperature has changed since 1940.

We will move from a global time series to spatial patterns:

1. Load and inspect the gridded temperature dataset
2. Calculate a latitude-weighted global mean temperature anomaly
3. Visualize the long-term warming trend
4. Examine the early-2000s **surface warming hiatus**
5. Compare trends over different periods
6. Calculate the latitude-dependent warming pattern
7. Identify **polar amplification**
8. Explore how a global mean can hide important spatial differences

> **Dataset:** `Global_TAVG_Gridded_5deg.nc`  
> Berkeley Earth Global TAVG, a gridded land-and-ocean surface temperature reconstruction.


## 1. Imports and file path

The only data file needed for this notebook is:

```text
data/Global_TAVG_Gridded_5deg.nc
```

Because the grid is coarse, the dataset should be small enough to work comfortably in Binder.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

from pathlib import Path

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True

data_file = Path("../data/Global_TAVG_Gridded_5deg.nc")

ds = xr.open_dataset(data_file)
ds

## 2. Inspect the dataset

Before doing any analysis, look at the dimensions, coordinates, variables, and metadata.

This is an important habit when working with climate data: **never assume that every NetCDF file uses the same variable names or dimension names.**

In [ ]:
print("Dimensions:")
print(ds.dims)

print("\nCoordinates:")
print(list(ds.coords))

print("\nData variables:")
print(list(ds.data_vars))

print("\nGlobal attributes:")
for key, value in ds.attrs.items():
    print(f"{key}: {value}")

In [ ]:
# Berkeley Earth gridded products commonly use the variable name "temperature".
temperature = ds["temperature"]

print(temperature)
print("\nUnits:", temperature.attrs.get("units"))
print("Long name:", temperature.attrs.get("long_name"))

## 3. Restrict the analysis to 1940–present

Although the Berkeley Earth record begins earlier, we will use **1940 onward** so that the analysis overlaps naturally with the modern reanalysis era and gives us a long record for studying recent climate change.

In [ ]:
temperature_1940 = temperature.sel(time=slice(1940, None))
temperature_1940

## 4. Calculate the global mean temperature anomaly

A simple arithmetic average would give every latitude band equal weight.

But a 5° × 5° grid cell near the poles covers a much smaller physical area than one near the Equator.

For a regular latitude–longitude grid we therefore use:

\[
w(\phi) = \cos(\phi)
\]

where \(\phi\) is latitude.

In [ ]:
weights = np.cos(np.deg2rad(temperature_1940.latitude))

global_mean = temperature_1940.weighted(weights).mean(
    dim=("latitude", "longitude")
)

global_mean

## 5. Plot the global monthly temperature anomaly

First look at the monthly record. Weather and short-term climate variability produce substantial fluctuations around the long-term warming trend.

In [ ]:
global_mean.plot()

plt.axhline(0, linewidth=1)
plt.title("Global mean surface temperature anomaly")
plt.ylabel("Temperature anomaly (°C)")
plt.xlabel("Year")
plt.show()

### Exercise

**What do you notice?**

- Is the warming trend monotonic?
- Are individual years very different from the long-term trend?
- Can you identify periods when warming is faster or slower?
- Does a short period give the same impression as the full record?

## 6. Annual means and the long-term warming trend

Annual averaging removes much of the month-to-month variability and makes the long-term climate signal easier to see.

In [ ]:
annual_mean = global_mean.groupby("time.year").mean()

annual_mean.plot()

plt.axhline(0, linewidth=1)
plt.title("Annual global mean temperature anomaly")
plt.ylabel("Temperature anomaly (°C)")
plt.xlabel("Year")
plt.show()

## 7. The global warming hiatus

The term **warming hiatus** is commonly used for the period around the early 2000s when the rate of increase in global mean **surface temperature** temporarily slowed.

It does **not** mean that global warming stopped.

The climate system continued to gain energy, while internal variability and changes in ocean heat uptake affected how much warming appeared in near-surface air temperature.

Let's compare the long-term record with the period around **1998–2013**.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

annual_mean.plot(ax=ax, linewidth=2, label="Annual mean")

ax.axvspan(1998, 2013, alpha=0.2, label="1998–2013")
ax.set_title("Global temperature anomaly and the early-2000s warming hiatus")
ax.set_ylabel("Temperature anomaly (°C)")
ax.set_xlabel("Year")
ax.legend()

plt.show()

### Important interpretation

The hiatus is best understood as a **temporary slowdown in the rate of surface warming**, not a halt in the accumulation of heat by the Earth system.

> **Surface temperature is only one indicator of Earth's changing energy state.**

## 8. Calculate warming rates

A trend depends strongly on the period chosen.

We will calculate linear warming rates for several periods and express them in **°C per decade**.

In [ ]:
def trend_per_decade(da, start_year, end_year):
    # Linear trend in °C per decade.
    subset = da.sel(year=slice(start_year, end_year)).dropna("year")
    x = subset.year.values.astype(float)
    y = subset.values.astype(float)
    slope, intercept = np.polyfit(x, y, 1)
    return slope * 10

periods = {
    "1940–1997": (1940, 1997),
    "1998–2013": (1998, 2013),
    "2014–present": (2014, int(annual_mean.year.max())),
    "1940–present": (1940, int(annual_mean.year.max())),
}

for label, (start, end) in periods.items():
    print(f"{label}: {trend_per_decade(annual_mean, start, end):.3f} °C/decade")

### Exercise

Compare the rates.

**Question:** Why can choosing a short time period give a misleading impression of the long-term climate trend?

Try calculating the trend for a few additional periods.

## 9. Running mean

A running mean suppresses short-term variability and emphasizes slower changes.

Here we use a **10-year running mean**.

In [ ]:
running_mean = annual_mean.rolling(year=10, center=True).mean()

fig, ax = plt.subplots(figsize=(11, 5))

annual_mean.plot(ax=ax, alpha=0.45, label="Annual mean")
running_mean.plot(ax=ax, linewidth=2.5, label="10-year running mean")

ax.set_title("Global temperature: annual variability and long-term change")
ax.set_ylabel("Temperature anomaly (°C)")
ax.set_xlabel("Year")
ax.legend()

plt.show()

## 10. Where is warming strongest?

The global mean compresses the entire planet into one number.

But warming is **spatially heterogeneous**.

We can calculate a temperature trend at every grid cell and then map the result.

In [ ]:
period = temperature_1940.sel(time=slice(1940, None))

year_decimal = xr.DataArray(
    period["time"].dt.year + (period["time"].dt.month - 0.5) / 12,
    dims="time",
    coords={"time": period.time},
)

trend = xr.apply_ufunc(
    lambda y, x: np.polyfit(x, y, 1)[0] * 10,
    period,
    year_decimal,
    input_core_dims=[["time"], ["time"]],
    output_core_dims=[[]],
    vectorize=True,
    dask="parallelized",
    output_dtypes=[float],
)

trend

In [ ]:
trend.plot(
    figsize=(11, 5),
    cmap="RdBu_r",
    vmin=-0.2,
    vmax=0.2,
    cbar_kwargs={"label": "°C per decade"},
)

plt.title("Temperature trend, 1940–present")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()

### What does the map show?

Look for broad spatial contrasts:

- Are land areas warming at the same rate as oceans?
- Is warming uniform with latitude?
- Where are the largest warming trends?
- Are there regions with weak warming or local cooling?

These spatial differences are a central feature of climate change.

## 11. Zonal-mean warming and polar amplification

To make the latitude dependence clearer, calculate the **zonal-mean trend**: average the trend across all longitudes at each latitude.

This gives a simple latitude profile of warming.

In [ ]:
zonal_trend = trend.mean(dim="longitude")

zonal_trend.plot(linewidth=2)

plt.axhline(0, linewidth=1)
plt.title("Zonal-mean temperature trend")
plt.xlabel("Latitude")
plt.ylabel("°C per decade")
plt.show()

### Polar amplification

The latitude profile should reveal that warming is not spatially uniform.

In particular, **high northern latitudes show enhanced warming relative to the global mean**.

This is called **polar amplification**.

Several mechanisms contribute, including:

- loss of snow and sea ice → reduced surface albedo
- changes in atmospheric lapse rates
- changes in water vapor and clouds
- changes in poleward heat transport

The strongest Arctic warming is particularly pronounced in the cold season.

## 12. Compare polar and global warming

A useful diagnostic is to compare the mean trend over the Arctic with the global mean trend.

Here we use **60–90°N** as a simple Arctic latitude band.

In [ ]:
arctic = temperature_1940.sel(latitude=slice(60, 90))

arctic_weights = np.cos(np.deg2rad(arctic.latitude))

arctic_mean = arctic.weighted(arctic_weights).mean(
    dim=("latitude", "longitude")
)

arctic_annual = arctic_mean.groupby("time.year").mean()

global_trend = trend_per_decade(
    annual_mean,
    1940,
    int(annual_mean.year.max())
)

arctic_trend = trend_per_decade(
    arctic_annual,
    1940,
    int(arctic_annual.year.max())
)

print(f"Global trend: {global_trend:.3f} °C/decade")
print(f"Arctic trend: {arctic_trend:.3f} °C/decade")
print(f"Arctic / global trend ratio: {arctic_trend / global_trend:.2f}")

## 13. Final exercise: connect the mechanisms to the data

You have now seen three different ways of looking at the same climate record:

**Global mean**

→ How fast is the planet warming overall?

**Time evolution**

→ How does short-term variability affect our perception of the long-term trend?

**Spatial pattern**

→ Why does the warming response differ from place to place?

### Questions

1. Why does the early-2000s hiatus appear clearly in the global time series?
2. Does the hiatus imply that anthropogenic warming stopped?
3. Why is a latitude-weighted mean preferable to a simple grid-cell average?
4. What does the zonal-mean trend tell us about polar amplification?
5. Which mechanisms could explain enhanced Arctic warming?
6. Why might a global mean temperature anomaly hide important regional climate changes?

## Key takeaways

- **Global warming is a long-term trend, not a smooth year-to-year increase.**
- Short-term variability can produce periods of faster or slower surface warming.
- The early-2000s **warming hiatus** was a temporary slowdown in surface warming, not a cessation of global warming.
- Temperature change is **spatially heterogeneous**.
- High northern latitudes warm faster than the global average: **polar amplification**.
- A global mean is useful, but it can hide important regional and latitude-dependent changes.

---

**Data source:** Berkeley Earth Surface Temperature, Global TAVG gridded product. The Berkeley Earth documentation describes Global TAVG as a land-and-ocean temperature anomaly product; the exact release/version and reference period should be recorded with the file used in the course.